# 01 - Análisis Exploratorio, Preprocesamiento y Control Inicial
**Proyecto Final - Inteligencia Artificial II (UNCuyo)**  
**Alumno:** Enzo Palau (Legajo 13686)  

**Objetivo de este Notebook:**  
1. Configurar el entorno de cómputo acelerado por GPU (NVIDIA T4 en Google Colab / Kaggle).
2. Descargar y filtrar el dataset **NSynth** (familias acústicas: *brass, guitar, keyboard, string* en rango MIDI 36 a 84).
3. Aplicar un **balanceo estratificado** riguroso en el conjunto de entrenamiento (4.500 notas por familia, 18.000 en total).
4. Realizar el **Análisis de Datos y Espectrogramas** seleccionado para el informe y presentación:
   - **Análisis de Datos**: Mapa de calor bidimensional de cobertura tonal y dinámica (*Pitch vs Velocidad* por familia).
   - **Análisis de Espectrogramas**: Curvas de envolvente temporal de energía media (Perfil ADSR y curvas de decaimiento acústico por familia).
5. Diseñar la canalización de audio a espectrogramas mel de resolución $128 \times 128$ normalizados a $[-1, 1]$ vectorizada en GPU.
6. Ejecutar el **Control Inicial (Techo de Calidad Metodológico)**: Reconstrucción determinística con *Griffin-Lim*, medición de fidelidad espectral y reproductores audibles interactivos para las 4 familias.
7. Exportar los conjuntos preprocesados en archivos binarios comprimidos `.npz` (`train`, `valid`, `test`) y metadatos `.csv` listos para los modelos posteriores.


### ¿Qué se hace a continuación?
Se verifica la disponibilidad de aceleración por GPU (CUDA), se configuran las semillas pseudoaleatorias para garantizar reproducibilidad exacta (`SEED = 42`), y se reporta la memoria VRAM disponible.

### ¿Por qué se hace?
El cálculo de espectrogramas y las pruebas de inversión espectral (*Griffin-Lim*) son computacionalmente intensivas. Garantizar que PyTorch aproveche los núcleos CUDA de la GPU (como la T4 de Colab) y definir semillas fijas asegura que todas las particiones, muestreos y visualizaciones sean reproducibles.


In [ ]:
import os
import sys
import random
import numpy as np
import torch

# 1. Configurar semillas fijas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 2. Detección y selección de dispositivo GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Dispositivo configurado: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM total disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[ADVERTENCIA] No se detectó GPU. Se recomienda activar GPU en Colab (Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU).")


### ¿Qué se hace a continuación?
Se instalan los paquetes requeridos para el pipeline de audio y visualización (`kagglehub`, `soundfile`, `librosa`, `seaborn`) en caso de estar corriendo en una nueva sesión en la nube.

### ¿Por qué se hace?
Asegura que el entorno cuente con todas las librerías necesarias para descargar el dataset directamente desde Kaggle y procesar las señales de audio sin errores de importación.


In [ ]:
# Verificación e instalación automática de paquetes esenciales en Colab / Kaggle
try:
    import kagglehub
except ImportError:
    print("[INFO] Instalando kagglehub...")
    !pip install -q kagglehub

try:
    import soundfile
except ImportError:
    print("[INFO] Instalando soundfile...")
    !pip install -q soundfile

try:
    import librosa
except ImportError:
    print("[INFO] Instalando librosa...")
    !pip install -q librosa


### ¿Qué se hace a continuación?
Se define la estructura de directorios y la persistencia de datos. Si se ejecuta en Google Colab, se ofrece montar Google Drive de forma opcional para guardar los archivos `.npz` procesados.

### ¿Por qué se hace?
Permite persistir los espectrogramas generados para que los siguientes notebooks (el clasificador juez, el C-VAE y el Flow Matching) lean directamente los tensores preprocesados sin necesidad de volver a descargar ni convertir los archivos de audio en cada sesión.


In [ ]:
from pathlib import Path

# Detección de entorno Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[INFO] Ejecutando en Google Colab.")
    from google.colab import drive
    drive_mount_dir = Path('/content/drive')
    if not drive_mount_dir.exists():
        drive.mount('/content/drive')
    # Carpeta del proyecto en Drive
    PROJECT_DIR = drive_mount_dir / 'MyDrive' / 'IA2_Proyecto_Instrumentos'
else:
    print("[INFO] Ejecutando en entorno local o Kaggle.")
    PROJECT_DIR = Path('./data_processed')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR = PROJECT_DIR / 'raw'
PROCESSED_DATA_DIR = PROJECT_DIR / 'processed'

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Directorio de datos sin procesar: {RAW_DATA_DIR}")
print(f"[INFO] Directorio de datos procesados: {PROCESSED_DATA_DIR}")


### ¿Qué se hace a continuación?
Se descarga o localiza el dataset público **NSynth** (Engel et al., 2017) en formato WAV con sus metadatos JSON a través del paquete `kagglehub`.

### ¿Por qué se hace?
NSynth es la referencia estándar para síntesis de audio musical por notas individuales. Contiene grabaciones estandarizadas a 16 kHz de 4 segundos con anotaciones de familia, pitch MIDI, velocidad y fuente sonora.


In [ ]:
import kagglehub
import json
from pathlib import Path

# Dataset en Kaggle con audios WAV completos
KAGGLE_DATASET = "accelotron/nsynth-wav"

print(f"[INFO] Descargando o verificando dataset '{KAGGLE_DATASET}' con kagglehub...")
dataset_root_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print(f"[INFO] Dataset disponible en: {dataset_root_path}")

# Localizar carpetas internas
subdirs = [p for p in dataset_root_path.glob("**/*") if p.is_dir() and "nsynth-" in p.name.lower()]
print(f"[INFO] Subdirectorios encontrados: {[s.name for s in subdirs]}")


### ¿Qué se hace a continuación?
Se leen los archivos de metadatos JSON oficiales (`train`, `valid`, `test`) y se filtran las notas según los criterios del anteproyecto aprobado:
- **Fuente**: Únicamente notas acústicas (`instrument_source_str == 'acoustic'`).
- **Familias**: 4 familias representativas (`brass`, `guitar`, `keyboard`, `string`).
- **Rango de Altura**: Pitch MIDI 36 a 84 (Do 2 a Do 6, cubriendo el registro musical central audible).
- **Ruta de audio**: Vinculación unívoca al archivo `.wav` correspondiente en disco.

### ¿Por qué se hace?
El anteproyecto exige delimitar el problema a instrumentos acústicos reales en un rango tonal equilibrado para garantizar la viabilidad del entrenamiento bajo las cuotas de GPU gratuitas disponibles.


In [ ]:
import pandas as pd
import json
from pathlib import Path

# Familias objetivo acordadas en el anteproyecto
TARGET_FAMILIES = ['brass', 'guitar', 'keyboard', 'string']
PITCH_MIN = 36  # C2 (~65.4 Hz)
PITCH_MAX = 84  # C6 (~1046.5 Hz)

def load_nsynth_metadata(dataset_root: Path):
    dataset_root = Path(dataset_root)
    all_metadata = []
    
    # Mapeo de archivos JSON oficiales
    json_split_map = {
        'train': list(dataset_root.rglob('*train*.json')),
        'valid': list(dataset_root.rglob('*valid*.json')),
        'test': list(dataset_root.rglob('*test*.json'))
    }
    
    # Directorio común de audios
    audio_dirs = list(dataset_root.rglob('audio'))
    audio_dir = audio_dirs[0] if audio_dirs else dataset_root
    print(f'[INFO] Directorio de audio detectado: {audio_dir}')
    
    for split_name, files in json_split_map.items():
        if not files:
            print(f'[AVISO] No se encontró JSON para el split {split_name}')
            continue
            
        json_file = files[0]
        print(f'[INFO] Leyendo metadatos de {split_name}: {json_file.name}...')
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        for note_id, meta in data.items():
            # Filtrar por acústico, familia objetivo y rango de pitch
            if meta.get('instrument_source_str') == 'acoustic' and                meta.get('instrument_family_str') in TARGET_FAMILIES and                PITCH_MIN <= meta.get('pitch', 0) <= PITCH_MAX:
                
                audio_path = audio_dir / f'{note_id}.wav'
                all_metadata.append({
                    'note_id': note_id,
                    'split': split_name,
                    'family': meta.get('instrument_family_str'),
                    'family_id': TARGET_FAMILIES.index(meta.get('instrument_family_str')),
                    'instrument_id': meta.get('instrument', 0),
                    'pitch': meta.get('pitch'),
                    'velocity': meta.get('velocity'),
                    'audio_path': str(audio_path)
                })
                
    df = pd.DataFrame(all_metadata)
    return df

df_meta = load_nsynth_metadata(dataset_root_path)
print(f"[INFO] Total de notas acústicas filtradas: {len(df_meta)}")
print("\nDistribución inicial por split:")
print(df_meta['split'].value_counts())
print("\nDistribución inicial por familia:")
print(df_meta['family'].value_counts())


### ¿Qué se hace a continuación?
Se aplica un **balanceo estratificado** sobre la partición de entrenamiento (*train*) seleccionando exactamente 4.500 notas por familia musical mediante muestreo determinístico (`SEED = 42`), preservando intactas las particiones oficiales de validación y prueba.

### ¿Por qué se hace?
En el dataset original, la familia *string* tiene más de 16.000 notas mientras *keyboard* cuenta con ~4.800 notas. Si entrenáramos con ese desbalance, el generador y el clasificador estarían fuertemente sesgados hacia las cuerdas. Al balancear a 4.500 muestras por familia (18.000 notas en total para entrenamiento):
1. Cada familia tiene una probabilidad de aparición idéntica del 25%.
2. El tamaño en memoria de los tensores preprocesados se reduce a ~1.1 GB, permitiendo un entrenamiento ultra-rápido en la GPU T4 de Colab sin riesgo de saturar la RAM.


In [ ]:
import pandas as pd

# Balanceo estratificado del conjunto de entrenamiento
TARGET_SAMPLES_PER_FAMILY = 4500  # Acorde a la familia minoritaria (keyboard)
SEED = 42

df_train = df_meta[df_meta['split'] == 'train']
df_valid = df_meta[df_meta['split'] == 'valid']
df_test = df_meta[df_meta['split'] == 'test']

# Muestreo balanceado por familia en train con semilla fija
df_train_balanced = (
    df_train.groupby('family', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), TARGET_SAMPLES_PER_FAMILY), random_state=SEED), include_groups=True)
    .reset_index(drop=True)
)

# Reemplazar df_meta con el conjunto curado y balanceado
df_meta = pd.concat([df_train_balanced, df_valid, df_test], ignore_index=True)

print("[INFO] Distribución en TRAIN luego del balanceo:")
print(df_train_balanced['family'].value_counts())
print(f"\nTotal general curado: {len(df_meta)} notas (Train: {len(df_train_balanced)}, Valid: {len(df_valid)}, Test: {len(df_test)})")


### ¿Qué se hace a continuación?
Se generan visualizaciones estadísticas globales del subconjunto curado:
1. Conteo equilibrado por familia en *train* (4.500 c/u).
2. Histograma de distribución de alturas musicales (*pitches* MIDI 36 a 84).
3. Conteo de muestras por partición oficial (*train*, *valid*, *test*).
4. Distribución de velocidades MIDI (niveles de dinámica sonora: 25, 50, 75, 100, 127).

### ¿Por qué se hace?
Permite certificar empíricamente el balance alcanzado y verificar la cobertura equitativa de todos los registros antes de proceder a la extracción de espectrogramas.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

TARGET_FAMILIES = ['brass', 'guitar', 'keyboard', 'string']
PITCH_MIN = 36
PITCH_MAX = 84

sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(15, 9), dpi=120)

# 1. Balance de Familias en Train
palette_families = sns.color_palette("deep", len(TARGET_FAMILIES))
sns.countplot(data=df_meta[df_meta['split'] == 'train'], x='family', order=TARGET_FAMILIES, palette=palette_families, ax=axes[0, 0])
axes[0, 0].set_title("Distribución por Familia en TRAIN (Balanceado: 4.500 c/u)", fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel("Familia de Instrumento")
axes[0, 0].set_ylabel("Cantidad de Muestras")
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Distribución de Altura Musical (Pitch MIDI)
sns.histplot(data=df_meta[df_meta['split'] == 'train'], x='pitch', hue='family', multiple='stack', 
             bins=PITCH_MAX - PITCH_MIN + 1, palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title(f"Distribución de Pitch MIDI en TRAIN ({PITCH_MIN} a {PITCH_MAX})", fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel("Pitch MIDI (Nota Musical)")
axes[0, 1].set_ylabel("Frecuencia")

# 3. Distribución por Split Oficial
split_order = ['train', 'valid', 'test']
sns.countplot(data=df_meta, x='split', order=split_order, palette='magma', ax=axes[1, 0])
axes[1, 0].set_title("Distribución por Partición Oficial de Datos", fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel("Split")
axes[1, 0].set_ylabel("Total de Notas")
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Distribución de Velocidad MIDI (Dinámica)
sns.countplot(data=df_meta[df_meta['split'] == 'train'], x='velocity', hue='family', palette=palette_families, ax=axes[1, 1])
axes[1, 1].set_title("Distribución de Velocidad MIDI (Intensidad / Timbre)", fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel("Velocidad MIDI")
axes[1, 1].set_ylabel("Cantidad")

plt.tight_layout()
plt.show()


### ¿Qué se hace a continuación?
Se analiza la **cobertura cruzada bidimensional del espacio de control acústico (*Pitch vs Velocidad*) para cada una de las 4 familias musicales** mediante mapas de calor (*heatmaps* 2D):
1. Para cada familia (*brass, guitar, keyboard, string*), se tabula la matriz de contingencia entre las notas tonales (Pitch MIDI 36 a 84, eje X) y los niveles de intensidad dinámica (Velocidad MIDI 25, 50, 75, 100, 127, eje Y).
2. Se grafica una cuadrícula de $2 \times 2$ mapas de calor comparativos con escala de color unificada y balanceada.

### ¿Por qué se hace?
Para el informe escrito de la tesis y las diapositivas de la presentación final, este gráfico aporta una justificación metodológica crítica:
- **Demostrar la ausencia de 'puntos ciegos'**: Verifica empíricamente que el espacio condicional $(c_{\text{pitch}}, c_{\text{velocity}})$ está cubierto de forma densa y regular en todas las familias, evitando regiones vacías que desestabilicen el entrenamiento de los modelos generativos.
- **Caracterizar el balance tímbrico-dinámico**: Asegura que el generador condicional (C-VAE y Flow Matching) aprenda a sintetizar notas tanto pianissimo (vel 25) como fortissimo (vel 127) de manera consistente en todo el registro instrumental.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Garantizar definición de familias objetivo
TARGET_FAMILIES = ['brass', 'guitar', 'keyboard', 'string']
train_df = df_meta[df_meta['split'] == 'train']

fig, axes = plt.subplots(2, 2, figsize=(16, 8), dpi=120)
axes = axes.flatten()

# Paleta y límites comunes para comparación justa
cmap_choice = 'YlGnBu'

for idx, fam in enumerate(TARGET_FAMILIES):
    fam_subset = train_df[train_df['family'] == fam]
    # Matriz cruzada de Pitch (columnas) vs Velocidad (filas)
    cross_tab = pd.crosstab(fam_subset['velocity'], fam_subset['pitch'])
    
    # Reindexar para asegurar que todos los pitches (36 a 84) y velocidades estén presentes
    all_pitches = list(range(36, 85))
    all_velocities = [25, 50, 75, 100, 127]
    cross_tab = cross_tab.reindex(index=all_velocities, columns=all_pitches, fill_value=0)
    
    sns.heatmap(cross_tab, ax=axes[idx], cmap=cmap_choice, cbar=(idx % 2 == 1),
                cbar_kws={'label': 'Muestras'} if (idx % 2 == 1) else None)
    
    axes[idx].set_title(f"Familia: {fam.upper()} (Total Train: {len(fam_subset):,} notas)", fontsize=11, fontweight='bold')
    axes[idx].set_xlabel("Pitch MIDI (Nota Musical: 36 a 84)", fontsize=10)
    axes[idx].set_ylabel("Velocidad MIDI (Dinámica)", fontsize=10)
    axes[idx].invert_yaxis()  # Velocidades menores abajo, mayores arriba

plt.suptitle("Mapa de Calor 2D: Cobertura Cruzada de Pitch vs Velocidad por Familia en TRAIN", 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### ¿Qué se hace a continuación?
Se implementa la clase `AudioProcessorGPU` basada en `torchaudio` optimizada para ejecutarse en los núcleos CUDA de la GPU:
- Frecuencia de muestreo: $16.000$ Hz (estándar de NSynth).
- Parámetros de espectrograma: `n_fft = 2048`, `win_length = 2048`, `hop_length = 500`, `n_mels = 128`.
- Duración fija de señal: $64.000$ muestras ($4$ segundos a $16$ kHz). Con salto de $500$ muestras, genera exactamente una cuadrícula cuadrada de **$128 \times 128$** píxeles de resolución en escala mel de potencia.
- Escala y normalización: Conversión a decibeles logarítmicos (`AmplitudeToDB` con rango dinámico de $80$ dB) y reescalado lineal al intervalo $[-1, 1]$.

### ¿Por qué se hace?
El procesamiento en GPU mediante tensores de PyTorch reduce el tiempo de conversión de horas a escasos minutos para decenas de miles de notas. Además, la resolución cuadrada fija de $128 \times 128$ normalizada a $[-1, 1]$ es el estándar óptimo para alimentar modelos convolucionales generativos (C-VAE con arquitectura UNet/ResNet y Flow Matching).


In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T

class AudioProcessorGPU:
    def __init__(self, sample_rate=16000, n_fft=2048, hop_length=500, n_mels=128, target_length=64000, device='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.target_length = target_length
        
        # Transformación espectrograma mel en GPU (escala de potencia = 2.0)
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=self.n_fft,
            win_length=self.n_fft,
            hop_length=self.hop_length,
            n_mels=self.n_mels,
            power=2.0
        ).to(self.device)
        
        # Conversión a escala logarítmica (decibeles) con rango dinámico de 80 dB
        self.amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80.0).to(self.device)

    def wav_to_mel(self, waveform: torch.Tensor) -> torch.Tensor:
        """
        Convierte forma de onda (B, 1, T) o (1, T) a espectrograma mel logarítmico (B, 1, 128, 128) normalizado a [-1, 1].
        """
        waveform = waveform.to(self.device)
        if waveform.dim() == 2:
            waveform = waveform.unsqueeze(1)  # Asegurar dimensión de canal (B, 1, T)
            
        # Ajustar longitud exacta a target_length (64.000 muestras = 4s)
        if waveform.shape[-1] < self.target_length:
            pad = self.target_length - waveform.shape[-1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
        else:
            waveform = waveform[..., :self.target_length]
            
        # Calcular Mel Spectrogram de potencia y pasar a decibeles
        mel = self.mel_spectrogram(waveform)
        mel_db = self.amplitude_to_db(mel)
        
        # Recortar o ajustar exactamente a 128 frames temporales
        mel_db = mel_db[..., :128]
        
        # Normalizar al rango [-1, 1] asumiendo un rango dinámico de 80 dB
        mel_norm = (mel_db + 40.0) / 40.0
        mel_norm = torch.clamp(mel_norm, -1.0, 1.0)
        
        return mel_norm

# Instanciar el procesador en el dispositivo disponible
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
audio_proc = AudioProcessorGPU(device=device_type)
print(f"[INFO] Procesador de audio inicializado exitosamente en: {audio_proc.device}")


### ¿Qué se hace a continuación?
Se analiza la **Dinámica Temporal y la Curva de Decaimiento de Energía Acústica (Envolvente ADSR)**:
1. Para cada una de las 4 familias musicales (*brass, guitar, keyboard, string*), se extrae un lote representativo de notas acústicas procesadas en GPU.
2. Se computa la envolvente de energía promedio frame a frame a lo largo de los 128 instantes temporales (4 segundos de audio).
3. Se grafican las 4 curvas comparativas de energía media con bandas sombreadas de desviación estándar ($\pm 0.5\,\sigma$) y anotaciones de las fases acústicas fundamentales (Ataque, Sostenimiento, Decaimiento y Liberación).

### ¿Por qué se hace?
Las familias de instrumentos no solo difieren en su contenido armónico (timbre frecuencial), sino de manera crítica en su **comportamiento temporal intrínseco**:
- **Cuerda punteada / percutida (*guitar*, *keyboard*)**: Ataque explosivo cuasi-instantáneo ($t pprox 0$ s) seguido de un decaimiento exponencial monótono decreciente (*Decay & Release*).
- **Instrumentos de excitación continua (*brass*, *string*)**: Fase de sostenimiento prolongado y estable (*Sustain*) donde la energía acústica se mantiene constante casi hasta los 3 segundos antes de extinguirse.
Esta figura es estelar para la tesis porque explica con física acústica por qué los generadores condicionales (C-VAE y Flow Matching) deben capturar correlaciones temporales tanto de corto como de largo alcance.


In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import matplotlib.pyplot as plt

# Parámetros y familias
TARGET_FAMILIES = ['brass', 'guitar', 'keyboard', 'string']
SEED = 42
N_SAMPLES_PER_FAM = 120

colors = {'brass': '#e74c3c', 'guitar': '#3498db', 'keyboard': '#2ecc71', 'string': '#9b59b6'}
time_axis_sec = np.linspace(0.0, 4.0, 128)

print(f"[INFO] Computando envolventes temporales en GPU ({N_SAMPLES_PER_FAM} muestras por familia)...")

fig, ax = plt.subplots(figsize=(13, 6), dpi=120)

for fam in TARGET_FAMILIES:
    # Muestrear notas representativas en train
    fam_samples = (
        df_meta[(df_meta['split'] == 'train') & (df_meta['family'] == fam)]
        .sample(n=min(len(df_meta[df_meta['family'] == fam]), N_SAMPLES_PER_FAM), random_state=SEED)
    )
    
    # Cargar y apilar audios en memoria
    loaded_wavs = []
    for p in fam_samples['audio_path']:
        wav, sr = torchaudio.load(p)
        if sr != 16000:
            wav = T.Resample(sr, 16000)(wav)
        loaded_wavs.append(wav)
        
    batch_wav = torch.stack(loaded_wavs, dim=0) # (N, 1, 64000)
    
    # Procesar espectrogramas en GPU
    with torch.no_grad():
        batch_mel = audio_proc.wav_to_mel(batch_wav).cpu().numpy().squeeze(1) # (N, 128, 128)
        
    # Magnitud media colapsando las bandas de frecuencia mel (128) -> vector temporal de 128 frames
    temporal_profile = np.mean(batch_mel, axis=1) # (N, 128)
    mean_energy = np.mean(temporal_profile, axis=0) # (128,)
    std_energy = np.std(temporal_profile, axis=0)   # (128,)
    
    # Graficar curva y dispersión
    ax.plot(time_axis_sec, mean_energy, label=f"{fam.capitalize()} (Media)", color=colors[fam], lw=2.5)
    ax.fill_between(time_axis_sec, mean_energy - 0.5 * std_energy, mean_energy + 0.5 * std_energy, 
                    color=colors[fam], alpha=0.15)

# Anotaciones físicas ADSR en el gráfico
ax.axvline(x=0.08, color='black', linestyle=':', alpha=0.6)
ax.text(0.10, 0.25, 'Ataque Rápido
(Percusión / Punteo)', fontsize=9, fontweight='bold', color='#2c3e50')

ax.axvline(x=3.0, color='gray', linestyle='--', alpha=0.7)
ax.text(3.02, -0.65, 'Liberación de Nota
(Release t ~ 3.0s)', fontsize=9, fontweight='bold', color='gray')

ax.set_title("Curvas de Envolvente Temporal de Energía Media (Perfil ADSR por Familia Acústica)", 
             fontsize=12, fontweight='bold')
ax.set_xlabel("Tiempo Transcurrido (segundos)", fontsize=11)
ax.set_ylabel("Magnitud Espectral Normalizada [-1, 1]", fontsize=11)
ax.set_xlim(0, 4.0)
ax.set_ylim(-1.0, 0.4)
ax.legend(title="Familia Acústica", loc='lower left', frameon=True, fontsize=10)
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()


### ¿Qué se hace a continuación?
Se implementa el módulo de inversión determinística **Griffin-Lim** para el **Control Inicial (Techo de Comparación Metodológico)**:
$$\text{Audio Real} \longrightarrow \text{Espectrograma Mel} \longrightarrow \text{Espectrograma Lineal} \longrightarrow \text{Griffin-Lim} \longrightarrow \text{Audio Reconstruido}$$
Se utiliza `librosa.feature.inverse.mel_to_audio` configurado con escala de potencia cuadrática (`power=2.0`), `n_iter=120` y normalización de amplitud de pico para garantizar una reconstrucción acústicamente nítida y audible.

### ¿Por qué se hace?
El anteproyecto establece este paso como el **límite superior teórico de fidelidad**:
Al convertir audio a magnitudes mel se descarta por completo la información de fase de Fourier. Griffin-Lim aproxima esa fase de manera iterativa. Por tanto, ningún modelo generativo (C-VAE o Flow Matching) que opere sobre espectrogramas mel podrá sonar mejor que el audio obtenido en este control inicial. Evaluar este techo nos da la referencia cuantitativa y cualitativa de pérdida atribuible al método de representación.


In [ ]:
import librosa
import numpy as np

class GriffinLimVocoder:
    def __init__(self, sample_rate=16000, n_fft=2048, hop_length=500, n_mels=128, n_iter=120):
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.n_iter = n_iter

    def mel_norm_to_audio(self, mel_norm: np.ndarray) -> np.ndarray:
        """
        Revierte la normalización [-1, 1] a decibeles, pasa a potencia lineal y aplica Griffin-Lim.
        """
        # 1. Desnormalizar de [-1, 1] a decibeles [-80 dB, 0 dB]
        mel_db = (mel_norm * 40.0) - 40.0
        # 2. Deshacer escala logarítmica a potencia lineal (power=2.0)
        mel_power = librosa.db_to_power(mel_db, ref=1.0)
        
        # 3. Inversión Mel a Audio mediante Griffin-Lim
        wav_rec = librosa.feature.inverse.mel_to_audio(
            mel_power,
            sr=self.sample_rate,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            power=2.0,
            n_iter=self.n_iter
        )
        
        # 4. Normalización de amplitud para escucha óptima
        max_val = np.max(np.abs(wav_rec))
        if max_val > 1e-4:
            wav_rec = wav_rec / max_val * 0.95
            
        return wav_rec

vocoder = GriffinLimVocoder(sample_rate=16000, n_fft=2048, hop_length=500, n_mels=128, n_iter=120)
print("[INFO] Vocoder Griffin-Lim configurado exitosamente.")


### ¿Qué se hace a continuación?
Se ejecuta la prueba de **Control Inicial interactiva con reproductores de audio para las 4 familias** (`brass`, `guitar`, `keyboard`, `string`):
1. Se carga una nota real de cada familia y se obtiene su espectrograma mel $128 \times 128$.
2. Se reconstruye la señal de audio mediante el vocoder determinístico Griffin-Lim.
3. Se calcula la métrica objetiva de distorsión espectral:
   - **Spectral Convergence (SC)**: $\frac{\|S_{\text{orig}} - S_{\text{rec}}\|_F}{\|S_{\text{orig}}\|_F}$
4. Se grafican las formas de onda comparativas y se despliegan reproductores de audio interactivos (`IPython.display.Audio`) para escuchar y contrastar Original vs Reconstruido en cada familia.

### ¿Por qué se hace?
Permite al estudiante y a los evaluadores comprobar la fidelidad perceptual alcanzable y fijar las métricas de error de referencia que se reportarán en la tabla comparativa del informe final.


In [ ]:
import librosa
import torch
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

TARGET_FAMILIES = ['brass', 'guitar', 'keyboard', 'string']

# Seleccionar una muestra representativa de cada familia en train
sample_control_df = df_meta[df_meta['split'] == 'train'].groupby('family').first().reset_index()

fig, axes = plt.subplots(4, 2, figsize=(14, 11), dpi=120)
reconstructed_audios = {}

for idx, row in sample_control_df.iterrows():
    fam = row['family']
    # 1. Cargar audio original a 16 kHz
    y_orig, sr = librosa.load(row['audio_path'], sr=16000, mono=True)
    if len(y_orig) > 64000:
        y_orig = y_orig[:64000]
    else:
        y_orig = np.pad(y_orig, (0, 64000 - len(y_orig)))
        
    # 2. Obtener espectrograma mel normalizado [-1, 1] en GPU
    with torch.no_grad():
        mel_t = audio_proc.wav_to_mel(torch.tensor(y_orig, dtype=torch.float32).unsqueeze(0))
        mel_norm = mel_t.squeeze().cpu().numpy()
        
    # 3. Inversión determinística con Griffin-Lim
    y_rec = vocoder.mel_norm_to_audio(mel_norm)
    reconstructed_audios[fam] = (y_orig, y_rec)
    
    # 4. Métrica de Spectral Convergence (SC)
    S_orig = np.abs(librosa.stft(y_orig, n_fft=2048, hop_length=500))
    S_rec = np.abs(librosa.stft(y_rec, n_fft=2048, hop_length=500))
    spectral_convergence = np.linalg.norm(S_orig - S_rec, 'fro') / np.linalg.norm(S_orig, 'fro')
    
    # Visualización lado a lado
    ax_mel = axes[idx, 0]
    ax_mel.imshow(mel_norm, origin='lower', aspect='auto', cmap='magma', vmin=-1.0, vmax=1.0)
    ax_mel.set_title(f"Mel: {fam.upper()} (Pitch {row['pitch']}) | Spectral Convergence: {spectral_convergence:.4f}", 
                     fontsize=10, fontweight='bold')
    ax_mel.set_ylabel("Mel Bins")
    if idx == 3:
        ax_mel.set_xlabel("Tiempo (Frames)")
        
    ax_wav = axes[idx, 1]
    ax_wav.plot(y_orig[:4000], label='Original', color='#2980b9', alpha=0.8, lw=1.2)
    ax_wav.plot(y_rec[:4000], label='Griffin-Lim', color='#e67e22', alpha=0.8, lw=1.2)
    ax_wav.set_title(f"Forma de Onda (primeros 250ms)", fontsize=10)
    ax_wav.set_ylabel("Amplitud")
    ax_wav.legend(loc='upper right', fontsize=8)
    if idx == 3:
        ax_wav.set_xlabel("Muestras de Audio")

plt.tight_layout()
plt.show()

# Reproductores interactivos para las 4 familias
print("=" * 70)
print("CONTROL INICIAL AUDIBLE: COMPARATIVA ORIGINAL VS GRIFFIN-LIM")
print("=" * 70)

for fam in TARGET_FAMILIES:
    y_orig, y_rec = reconstructed_audios[fam]
    print(f"\n--- FAMILIA: {fam.upper()} ---")
    print(f"1. Audio Original ({fam}):")
    ipd.display(ipd.Audio(y_orig, rate=16000))
    print(f"2. Audio Reconstruido Griffin-Lim ({fam}):")
    ipd.display(ipd.Audio(y_rec, rate=16000))


### ¿Qué se hace a continuación?
Se procesa el subconjunto curado completo de forma vectorizada por lotes en GPU y se exporta a archivos binarios comprimidos `.npz`:
- `nsynth_processed_train.npz` (18.000 espectrogramas balanceados de $1 \times 128 \times 128$, etiquetas de familia, pitch e instrumento).
- `nsynth_processed_valid.npz` (2.297 muestras oficiales).
- `nsynth_processed_test.npz` (771 muestras oficiales).
- `nsynth_metadata_filtered.csv` (Registro tabular con rutas y anotaciones).

### ¿Por qué se hace?
El cálculo de espectrogramas para ~21.000 notas se realiza una única vez. Guardar los tensores en formato binario comprimido `.npz` permite que los notebooks de:
1. **Entrenamiento del Juez Clasificador (`02_entrenamiento_juez_evaluador.ipynb`)**
2. **Generador C-VAE (`03_generador_cvae.ipynb`)**
3. **Generador Flow Matching (`04_generador_flow_matching.ipynb`)**
carguen directamente los arreglos en memoria en menos de 5 segundos sin tener que reabrir ni reprocesar los archivos WAV individuales en cada sesión.


In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class NSynthBatchDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, sr = torchaudio.load(row['audio_path'])
        if sr != 16000:
            wav = T.Resample(sr, 16000)(wav)
        return wav.squeeze(0), row['family_id'], row['pitch'], row['instrument_id']

def process_and_save_split(df_split, split_name, output_dir, batch_size=64):
    print(f"[INFO] Procesando split '{split_name}' ({len(df_split)} muestras)...")
    dataset = NSynthBatchDataset(df_split)
    # DataLoader con 2 workers para optimizar I/O
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    all_mels = []
    all_families = []
    all_pitches = []
    all_instruments = []
    
    with torch.no_grad():
        for wavs, families, pitches, instrs in tqdm(loader, desc=f"Procesando {split_name}"):
            # Procesamiento vectorizado en GPU
            mels = audio_proc.wav_to_mel(wavs)  # Forma: (B, 1, 128, 128)
            all_mels.append(mels.cpu().numpy().astype(np.float32))
            all_families.append(families.numpy().astype(np.int64))
            all_pitches.append(pitches.numpy().astype(np.int64))
            all_instruments.append(instrs.numpy().astype(np.int64))
            
    mels_arr = np.concatenate(all_mels, axis=0)
    families_arr = np.concatenate(all_families, axis=0)
    pitches_arr = np.concatenate(all_pitches, axis=0)
    instrs_arr = np.concatenate(all_instruments, axis=0)
    
    output_file = output_dir / f"nsynth_processed_{split_name}.npz"
    np.savez_compressed(
        output_file,
        mels=mels_arr,
        families=families_arr,
        pitches=pitches_arr,
        instruments=instrs_arr
    )
    print(f"[INFO] Guardado exitoso: {output_file} | Forma: {mels_arr.shape}")

# Procesar cada partición
for split in ['train', 'valid', 'test']:
    df_s = df_meta[df_meta['split'] == split]
    if len(df_s) > 0:
        process_and_save_split(df_s, split, PROCESSED_DATA_DIR, batch_size=64)

# Guardar metadatos consolidados
csv_path = PROCESSED_DATA_DIR / "nsynth_metadata_filtered.csv"
df_meta.to_csv(csv_path, index=False)
print(f"[INFO] Metadatos guardados en: {csv_path}")


### Conclusiones del Análisis Exploratorio y Preprocesamiento
Con la ejecución de este notebook se alcanzaron los siguientes hitos:
1. **Curación y Balance Estratificado**: Selección de notas acústicas en las 4 familias objetivo (*brass, guitar, keyboard, string*) en el rango MIDI 36 a 84, balanceando *train* a 4.500 notas por familia (18.000 notas en total).
2. **Análisis de Datos y Espectrogramas para la Tesis**:
   - **Mapa de Calor 2D (Pitch vs Velocidad)**: Comprobación de cobertura densa y regular del espacio de condicionamiento acústico.
   - **Envolventes Temporales ADSR**: Caracterización del comportamiento temporal físico de cada familia (ataque instantáneo y decaimiento en instrumentos percutidos vs sostenimiento plano en instrumentos frotados/soplados).
3. **Validación del Control Inicial (Techo Griffin-Lim)**: Medición de convergencia espectral y verificación auditiva en las 4 familias.
4. **Generación y Persistencia de Tensores**: Almacenamiento de espectrogramas $128 \times 128$ normalizados a $[-1, 1]$ en archivos `.npz` comprimidos.

**Siguiente paso**: Ejecutar **`02_entrenamiento_juez_evaluador.ipynb`** para entrenar el clasificador de familias que actuará como juez objetivo de los generadores.
